In [ ]:
# ==========================================
# 1. SETUP & LIBRARY
# ==========================================
try:
    import segmentation_models_pytorch as smp
except ImportError:
    !pip install segmentation_models_pytorch
    import segmentation_models_pytorch as smp

import os
import cv2
import numpy as np
import pandas as pd
import random
from glob import glob
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import KFold


In [ ]:
# ==========================================
# 2. KONFIGURASI
# ==========================================
class Config:
    SEED = 42
    # Jika GPU memory cukup, naikkan ke 320 atau 384 untuk deteksi lubang kecil lebih baik
    IMG_SIZE = 256        
    BATCH_SIZE = 16      
    EPOCHS = 25           
    LR = 2e-4             
    N_FOLDS = 5           
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    NUM_WORKERS = 2
    
    # Path (Sesuaikan dengan path di Google Drive-mu)
    TRAIN_IMG_DIR = "/content/drive/MyDrive/ARA-ITS/dataset/train/images"
    TRAIN_MASK_DIR = "/content/drive/MyDrive/ARA-ITS/dataset/train/mask"
    TEST_IMG_DIR = "/content/drive/MyDrive/ARA-ITS/dataset/test/images"

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(Config.SEED)


In [ ]:
# ==========================================
# 3. UTILITIES
# ==========================================
def encode_rle(mask: np.ndarray, pos_value: int = 255) -> str:
    binary = (mask == pos_value).astype(np.uint8)
    pixels = binary.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(str(x) for x in runs)

def dice_coef(y_pred, y_true, smooth=1e-7):
    y_pred = y_pred.sigmoid()
    y_pred = (y_pred > 0.5).float()
    intersection = (y_pred * y_true).sum()
    return (2. * intersection + smooth) / (y_pred.sum() + y_true.sum() + smooth)

In [ ]:
# ==========================================
# 4. DATASET
# ==========================================
class PotholeDataset(Dataset):
    def __init__(self, img_paths, mask_paths=None, transform=None):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.mask_paths:
            mask_path = self.mask_paths[idx]
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask = np.where(mask > 127, 1.0, 0.0) # Binary 0/1
            
            if self.transform:
                augmented = self.transform(image=image, mask=mask)
                image = augmented['image']
                mask = augmented['mask']
            
            mask = mask.unsqueeze(0)
            return image, mask
        else:
            if self.transform:
                augmented = self.transform(image=image)
                image = augmented['image']
            return image, os.path.basename(img_path)

def get_transforms(data):
    if data == 'train':
        return A.Compose([
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5), # Tambahan rotasi 90 saat training
            A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.5),
            A.OneOf([
                A.RandomBrightnessContrast(p=0.5),
                A.HueSaturationValue(p=0.5),
            ], p=0.5),
            A.Normalize(),
            ToTensorV2(),
        ])
    elif data == 'valid':
        return A.Compose([
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
            A.Normalize(),
            ToTensorV2(),
        ])


In [ ]:
# ==========================================
# 5. LOSS & MODEL
# ==========================================
class BCEDiceLoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(BCEDiceLoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, inputs, targets, smooth=1):
        bce = self.bce(inputs, targets)
        inputs_sig = torch.sigmoid(inputs)
        inputs_flat = inputs_sig.view(-1)
        targets_flat = targets.view(-1)
        intersection = (inputs_flat * targets_flat).sum()
        dice = 1 - (2.*intersection + smooth)/(inputs_flat.sum() + targets_flat.sum() + smooth)
        return 0.5 * bce + 0.5 * dice


In [ ]:
# ==========================================
# 6. TRAINING LOOP (Standard 5-Fold)
# ==========================================
img_paths = np.array(sorted(glob(f"{Config.TRAIN_IMG_DIR}/*")))
mask_paths = np.array(sorted(glob(f"{Config.TRAIN_MASK_DIR}/*")))

kf = KFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=Config.SEED)
best_models = []

print(f"Mulai Training... (Pastikan path dataset benar)")

for fold, (train_idx, val_idx) in enumerate(kf.split(img_paths)):
    print(f"\n{'='*10} Fold {fold+1}/{Config.N_FOLDS} {'='*10}")
    
    train_ds = PotholeDataset(img_paths[train_idx], mask_paths[train_idx], transform=get_transforms('train'))
    val_ds = PotholeDataset(img_paths[val_idx], mask_paths[val_idx], transform=get_transforms('valid'))
    
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=Config.NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)
    
    model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1).to(Config.DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR)
    criterion = BCEDiceLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS, eta_min=1e-6)
    
    best_dice = 0.0
    best_model_path = f"unet_resnet34_fold{fold}.pth"
    
    for epoch in range(Config.EPOCHS):
        model.train()
        train_loss = 0
        for images, masks in tqdm(train_loader, desc="Train", leave=False):
            images, masks = images.to(Config.DEVICE), masks.to(Config.DEVICE).float()
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        model.eval()
        val_dice_score = 0
        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc="Val", leave=False):
                images, masks = images.to(Config.DEVICE), masks.to(Config.DEVICE).float()
                outputs = model(images)
                val_dice_score += dice_coef(outputs, masks).item()
        
        scheduler.step()
        train_loss /= len(train_loader)
        val_dice_score /= len(val_loader)
        
        if val_dice_score > best_dice:
            best_dice = val_dice_score
            torch.save(model.state_dict(), best_model_path)
            print(f"Epoch {epoch+1}: Dice improved to {best_dice:.4f}")
        else:
            print(f"Epoch {epoch+1}: Dice {val_dice_score:.4f}")

    best_models.append(best_model_path)
    del model, optimizer; torch.cuda.empty_cache()

In [ ]:
# ==========================================
# 7. INFERENCE WITH 8x TTA (Grandmaster Strategy)
# ==========================================
# Ini bagian terpenting untuk fix empty RLE dan boost score
print("\nMemulai Prediksi dengan 8x TTA...")

test_paths = sorted(glob(f"{Config.TEST_IMG_DIR}/*"))
test_ds = PotholeDataset(test_paths, transform=get_transforms('valid'))
test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)

# Load 5 Model
models = []
for path in best_models:
    m = smp.Unet(encoder_name="resnet34", in_channels=3, classes=1)
    m.load_state_dict(torch.load(path))
    m.to(Config.DEVICE)
    m.eval()
    models.append(m)

# Fungsi TTA Manual (8 views)
# 1. Original
# 2. Rot90, 3. Rot180, 4. Rot270
# 5. Flip Horizontal
# 6. FlipH + Rot90, 7. FlipH + Rot180, 8. FlipH + Rot270
def tta_inference(model, images):
    batch_size, c, h, w = images.shape
    preds = torch.zeros((batch_size, 1, h, w)).to(Config.DEVICE)
    
    # List transformasi: (flip?, k_rotasi)
    # k=0 (0deg), k=1 (90deg), k=2 (180deg), k=3 (270deg)
    transforms = [
        (False, 0), (False, 1), (False, 2), (False, 3), # Original + Rotations
        (True, 0),  (True, 1),  (True, 2),  (True, 3)   # H-Flip + Rotations
    ]
    
    for flip, k in transforms:
        # 1. Augment
        aug_img = images.clone()
        if flip:
            aug_img = torch.flip(aug_img, [3]) # Flip width
        if k > 0:
            aug_img = torch.rot90(aug_img, k, [2, 3]) # Rotasi di dimensi H, W
            
        # 2. Predict
        with torch.no_grad():
            p = model(aug_img)
            p = torch.sigmoid(p) # Probability 0-1
        
        # 3. Revert (Inverse Augment)
        # Revert rotation first: jika rotasi +90 (k=1), revertnya -90 (k=3)
        if k > 0:
            p = torch.rot90(p, 4-k, [2, 3]) 
        # Revert flip
        if flip:
            p = torch.flip(p, [3])
            
        preds += p
    
    return preds / len(transforms) # Average dari 8 view

# Post-processing yang lebih hati-hati
def remove_small_blobs(mask, min_size=30): # Turunkan min_size dari 100 ke 30!
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    new_mask = np.zeros_like(mask)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_size:
            new_mask[labels == i] = 255
    return new_mask

submission_data = []

for images, ids in tqdm(test_loader):
    images = images.to(Config.DEVICE)
    
    # Ensemble Models (5 Fold) + TTA (8 View) = Total 40 Votes per image!
    final_preds = torch.zeros((images.size(0), 1, Config.IMG_SIZE, Config.IMG_SIZE)).to(Config.DEVICE)
    
    for model in models:
        final_preds += tta_inference(model, images)
        
    final_preds /= len(models)
    final_preds = final_preds.cpu().numpy()
    
    for i in range(len(ids)):
        image_id = ids[i]
        pred_mask = final_preds[i, 0]
        
        # Resize ke ukuran asli (Wajib untuk submission akurat)
        # Kita baca ukuran asli dari file test
        original_img = cv2.imread(os.path.join(Config.TEST_IMG_DIR, image_id))
        h_orig, w_orig = original_img.shape[:2]
        pred_mask = cv2.resize(pred_mask, (w_orig, h_orig))
        
        # Threshold & Cleanup
        # Kita pakai 0.5 karena TTA sudah meratakan probability
        binary_mask = (pred_mask > 0.5).astype(np.uint8) * 255
        
        # Penting: Min size dikecilkan agar 'test_51' dkk tidak kosong jika memang ada dot kecil
        clean_mask = remove_small_blobs(binary_mask, min_size=30)
        
        if clean_mask.sum() == 0:
            rle_str = "" # Tetap kosong jika memang tidak ada apa-apa
        else:
            rle_str = encode_rle(clean_mask)
        
        submission_data.append([image_id, rle_str])

In [ ]:
df_sub = pd.DataFrame(submission_data, columns=['ImageId', 'rle'])
df_sub.to_csv('submission_tta_8x.csv', index=False)
print("Selesai! File 'submission_tta_8x.csv' siap submit.")